# Notebook 3: Infrastructure Telemetry Feature Engineering

**GPU Fleet Autopilot — Research & Simulation Suite**

This notebook computes the predictive feature matrix for failure prediction:
- **Exponentially Weighted Moving Averages (EWMA)**: 5-minute and 15-minute smoothers on temperature and ECC error arrivals
- **First-Order Derivatives**: Degradation slopes ($\frac{dT}{dt}$, $\frac{d\text{ECC}}{dt}$)
- **Physical Interaction Terms**: Die heat generation vs electrical power draw ($\frac{T}{100} \times \frac{P}{700}$)
- **Target Horizon Definition**: Ground-truth target `failure_in_next_2h`
- **Strict Leakage Controls**: Excluding terminal indicators (`xid_errors`, `status`, `health_score`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/sample_telemetry.csv")
df = df.sort_values(by=["gpu_id", "timestamp"]).reset_index(drop=True)
print(f"Loaded {len(df)} rows for feature computation.")

## 1. Compute Rolling EWMA and Slopes

In [ ]:
# Group by GPU to compute time-series rolling features
df["ewma_5m_temp"] = df.groupby("gpu_id")["dcgm_gpu_temp"].transform(lambda x: x.ewm(span=10).mean())
df["ewma_15m_temp"] = df.groupby("gpu_id")["dcgm_gpu_temp"].transform(lambda x: x.ewm(span=30).mean())

# ECC and NVLink error rates
df["ewma_5m_ecc_rate"] = df.groupby("gpu_id")["dcgm_ecc_sbe_volatile_total"].transform(lambda x: x.diff().fillna(0).ewm(span=10).mean())
df["ewma_15m_ecc_rate"] = df.groupby("gpu_id")["dcgm_ecc_sbe_volatile_total"].transform(lambda x: x.diff().fillna(0).ewm(span=30).mean())
df["nvlink_error_rate"] = df.groupby("gpu_id")["dcgm_nvlink_error_count"].transform(lambda x: x.diff().fillna(0))

# Interaction feature
df["temp_power_interaction"] = (df["dcgm_gpu_temp"] / 100.0) * (df["dcgm_power_usage"] / 700.0)

features = [
    "ewma_5m_temp",
    "ewma_15m_temp",
    "ewma_5m_ecc_rate",
    "ewma_15m_ecc_rate",
    "nvlink_error_rate",
    "performance_ratio",
    "dcgm_power_usage",
    "temp_power_interaction",
]

print("Engineered features:", features)
df[features + ["failure_in_next_2h"]].head()